# Model A convergence pilot, standalone Colab notebook

## Goal

This notebook tests whether the revised **centred, zero-sum parametrisation** removes the
convergence failure seen in Model A. It is deliberately narrow and **NON-REPORTABLE**:

- it reads the existing `02_contrasts.parquet` from the completed `d3_15model_v1` run;
- it never loads Chronos, generates signals, or recollects observations;
- it fits only Model A on a deterministic subset of 20 backgrounds per generator;
- it uses 4 chains with 500 tuning steps and 500 retained draws;
- it does not compute `log_likelihood`, LOO, sensitivity fits, PPCs, or an H1 verdict;
- it writes only to a new pilot directory and never overwrites the original run.

### How to run in Colab

1. Upload this notebook to Drive and open it with Colab.
2. A normal **CPU runtime** is sufficient. A GPU is not used by the default `nutpie` backend.
3. Choose **Runtime > Run all**.
4. The environment cell may deliberately restart the runtime once. After reconnection, choose
   **Runtime > Run all** again.
5. At the end, send the printed `PILOT ROUTE` and the diagnostic table.

Expected total time is normally 25 to 60 minutes with `nutpie`; a slow Colab CPU may require up
to roughly 90 minutes. The posterior is a small checkpoint because no observation-wise
`log_likelihood` array is created.


## Setup

### 1. Locate or clone the repository

This is copied from the operational Bayesian notebook. The clone supplies the locked environment,
the frozen 15-model registry, and the atomic checkpoint helpers.


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/FedericoSabbadini/patchAliasing.git"
MARKER = Path("chronos") / "bayesian" / "probe_lib.py"


def on_colab() -> bool:
    try:
        return importlib.util.find_spec("google.colab") is not None
    except ModuleNotFoundError:
        return False


IS_COLAB = on_colab()


def find_repo() -> Path:
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / MARKER).is_file():
            return candidate
    target = Path("/content/patchAliasing") if IS_COLAB else here / "patchAliasing"
    if not (target / MARKER).is_file():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(target)])
    if not (target / MARKER).is_file():
        raise FileNotFoundError(f"{MARKER} is missing from {target}")
    return target


REPO = find_repo()
BAYES_DIR = REPO / "chronos" / "bayesian"
if str(BAYES_DIR) not in sys.path:
    sys.path.insert(0, str(BAYES_DIR))

print("repository:", REPO)
print("modules   :", BAYES_DIR)
print("runtime   :", "Colab" if IS_COLAB else "local")


### 2. Install the locked environment

On a fresh Colab VM this installs the repository lock and deliberately restarts the process once,
because replacing NumPy or SciPy inside an already running kernel is unsafe. The restart state is
kept only in the temporary Colab VM. Outside Colab this cell does not modify the environment.


In [ ]:
import json
import shutil
import sysconfig
import tempfile
import time

RESTART_STATE_PATH = Path(tempfile.gettempdir()) / "patchaliasing_A_pilot_env_state.json"
MAX_RESTARTS = 2


def uv_executable() -> str:
    found = shutil.which("uv")
    if found:
        return found
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "uv"])
    candidates = [
        Path(sysconfig.get_path("scripts")) / ("uv.exe" if os.name == "nt" else "uv"),
        Path(sys.executable).parent / ("uv.exe" if os.name == "nt" else "uv"),
    ]
    for candidate in candidates:
        if candidate.is_file():
            return str(candidate)
    raise FileNotFoundError("uv was installed but its executable was not found")


def load_restart_state() -> dict:
    if RESTART_STATE_PATH.is_file():
        try:
            return json.loads(RESTART_STATE_PATH.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            pass
    return {"restarts": 0}


def save_restart_state(state: dict) -> None:
    RESTART_STATE_PATH.write_text(json.dumps(state), encoding="utf-8")


def clean_imports_are_healthy() -> bool:
    probe = subprocess.run(
        [
            sys.executable,
            "-c",
            "import numpy, scipy, pandas, pyarrow, arviz, pymc, nutpie, h5netcdf",
        ],
        capture_output=True,
        text=True,
    )
    if probe.returncode != 0:
        print(probe.stderr[-2500:])
    return probe.returncode == 0


def restart_colab(reason: str) -> None:
    print("=" * 78)
    print(f"RESTARTING THE COLAB RUNTIME: {reason}")
    print("This is deliberate. After reconnection choose Runtime > Run all again.")
    print("=" * 78)
    sys.stdout.flush()
    time.sleep(2)
    os.kill(os.getpid(), 9)


if not IS_COLAB:
    print("Local runtime: dependency installation skipped; using the active environment.")
else:
    state = load_restart_state()
    UV = uv_executable()

    if state["restarts"] == 0:
        with tempfile.TemporaryDirectory() as temporary:
            requirements = Path(temporary) / "requirements.locked.txt"
            subprocess.check_call(
                [
                    UV,
                    "export",
                    "--frozen",
                    "--no-dev",
                    "--no-emit-project",
                    "--no-hashes",
                    "--output-file",
                    str(requirements),
                ],
                cwd=REPO,
            )
            subprocess.check_call(
                [UV, "pip", "install", "--python", sys.executable, "--requirement", str(requirements)]
            )

        # Required for saving the small posterior checkpoint on Python 3.12+.
        subprocess.check_call(
            [UV, "pip", "install", "--python", sys.executable, "h5netcdf", "h5py"]
        )

        # Colab's preinstalled vision wheels can be ABI-incompatible with the locked torch.
        # They are not used by this pilot.
        subprocess.run(
            [sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio"],
            check=False,
            capture_output=True,
        )
        save_restart_state({"restarts": 1})
        restart_colab("locked environment installed")

    elif state["restarts"] == 1:
        if clean_imports_are_healthy():
            save_restart_state({"restarts": "verified"})
            print("Locked environment verified in the restarted runtime.")
        else:
            subprocess.check_call(
                [
                    UV,
                    "pip",
                    "install",
                    "--python",
                    sys.executable,
                    "--reinstall-package",
                    "numpy",
                    "--reinstall-package",
                    "scipy",
                ]
            )
            save_restart_state({"restarts": 2})
            restart_colab("NumPy/SciPy clean reinstall")
    else:
        if not clean_imports_are_healthy():
            raise RuntimeError(
                "The scientific imports are still broken after two restarts. Choose Runtime > "
                "Disconnect and delete runtime, reconnect to a fresh VM, and run all again."
            )
        save_restart_state({"restarts": "verified"})
        print("Locked environment verified.")


### 3. Parameters and Drive paths

The defaults point to the completed original run and to a separate pilot namespace. Do not point
`PILOT_ROOT` inside `SOURCE_ROOT`. To repeat the experiment with different settings, change
`PILOT_RUN_ID` rather than deleting or overwriting a completed pilot.


In [ ]:
from __future__ import annotations

import gc
import platform
import random
from importlib import metadata as importlib_metadata

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import xarray as xr
from IPython.display import display

import checkpointing as cp
import probe_lib as pl

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

SOURCE_RUN_ID = "d3_15model_v1"
PILOT_RUN_ID = "model_A_centred_zerosum_pilot_v1"

# Default scientific pilot settings. A local validation harness may override these with
# A_PILOT_VALIDATE_ONLY=1; Colab users should leave that variable unset.
VALIDATE_ONLY = os.environ.get("A_PILOT_VALIDATE_ONLY", "0") == "1"
PILOT_BG_PER_GENERATOR = 2 if VALIDATE_ONLY else 20
DRAWS = 25 if VALIDATE_ONLY else 500
TUNE = 25 if VALIDATE_ONLY else 500
CHAINS = 2 if VALIDATE_ONLY else 4
TARGET_ACCEPT = 0.90
PILOT_ESS_MIN = 10 if VALIDATE_ONLY else 400
RHAT_MAX = 1.01
NUTS_BACKEND = os.environ.get("A_PILOT_NUTS_BACKEND", "nutpie").strip().lower()

if NUTS_BACKEND not in {"nutpie", "pymc"}:
    raise ValueError("A_PILOT_NUTS_BACKEND must be 'nutpie' or 'pymc'")
if NUTS_BACKEND == "nutpie" and importlib.util.find_spec("nutpie") is None:
    print("nutpie is unavailable; falling back to PyMC's built-in NUTS sampler.")
    NUTS_BACKEND = "pymc"

if IS_COLAB:
    from google.colab import drive

    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    default_drive_root = Path("/content/drive/MyDrive/patchAliasing")
else:
    default_drive_root = BAYES_DIR / "_run"

DRIVE_ROOT = Path(os.environ.get("A_PILOT_DRIVE_ROOT", str(default_drive_root)))
SOURCE_ROOT = Path(
    os.environ.get("A_PILOT_SOURCE_ROOT", str(DRIVE_ROOT / "full" / SOURCE_RUN_ID))
)
PILOT_ROOT = Path(
    os.environ.get("A_PILOT_OUTPUT_ROOT", str(DRIVE_ROOT / "pilots" / PILOT_RUN_ID))
)
SOURCE_FILE = SOURCE_ROOT / "data" / "02_contrasts.parquet"

if SOURCE_ROOT == PILOT_ROOT or SOURCE_ROOT in PILOT_ROOT.parents:
    raise ValueError("PILOT_ROOT must be separate from, not inside, SOURCE_ROOT")

default_cores = max(1, min(2, CHAINS, os.cpu_count() or 1))
CORES = int(os.environ.get("A_PILOT_CORES", str(default_cores)))
if not 1 <= CORES <= CHAINS:
    raise ValueError(f"A_PILOT_CORES must be between 1 and {CHAINS}")
PLOT_STYLE = next(
    (name for name in ("arviz-whitegrid", "seaborn-v0_8-whitegrid") if name in plt.style.available),
    "default",
)
plt.style.use(PLOT_STYLE)

print("SOURCE_FILE        :", SOURCE_FILE)
print("PILOT_ROOT        :", PILOT_ROOT)
print("NON-REPORTABLE    :", True)
print("backgrounds/gen   :", PILOT_BG_PER_GENERATOR)
print("draws/tune/chains :", DRAWS, TUNE, CHAINS)
print("backend/cores     :", NUTS_BACKEND, CORES)
if VALIDATE_ONLY:
    print("LOCAL STRUCTURAL VALIDATION MODE: outputs are disposable and never scientific.")


## Steps

### 4. Load and validate the existing contrasts

The subset is selected before the `live` filter, using the same randomly selected background IDs
for every configuration within each generator. This preserves the paired design and gives each
of the 15 geometries coverage. Selection is deterministic under seed 42.


In [ ]:
REQUIRED_COLUMNS = {
    "model",
    "P",
    "S",
    "overlap",
    "generator",
    "bg_id",
    "f_lock",
    "phase_idx",
    "d",
    "live",
}

if not SOURCE_FILE.is_file():
    raise FileNotFoundError(
        f"Existing contrast table not found at {SOURCE_FILE}. Change SOURCE_RUN_ID or "
        "A_PILOT_SOURCE_ROOT to the completed run that contains data/02_contrasts.parquet."
    )

contrasts = pd.read_parquet(SOURCE_FILE, columns=sorted(REQUIRED_COLUMNS))
missing_columns = REQUIRED_COLUMNS - set(contrasts.columns)
if missing_columns:
    raise ValueError(f"02_contrasts.parquet is missing columns: {sorted(missing_columns)}")

expected_models = {pl.model_tag(P, S) for P, S in pl.DELIVERABLE3_MODELS}
observed_models = set(contrasts["model"].astype(str))
if observed_models != expected_models:
    raise ValueError(
        "The source is not the frozen 15-model design. "
        f"missing={sorted(expected_models - observed_models)}, "
        f"unexpected={sorted(observed_models - expected_models)}"
    )

expected_generators = set(pl.GENERATORS)
observed_generators = set(contrasts["generator"].astype(str))
if observed_generators != expected_generators:
    raise ValueError(
        f"Generator coverage mismatch: expected {sorted(expected_generators)}, "
        f"observed {sorted(observed_generators)}"
    )

if contrasts[["model", "generator", "bg_id", "f_lock", "phase_idx"]].duplicated().any():
    raise ValueError("Duplicate Model A contrast keys found in the source table")
if not np.isfinite(contrasts["d"].to_numpy(float)).all():
    raise ValueError("The source contrast d contains NaN or infinite values")

rng = np.random.default_rng(SEED)
selected_bg_ids = {}
for generator in sorted(expected_generators):
    by_model = []
    generator_rows = contrasts[contrasts["generator"] == generator]
    for model in sorted(expected_models):
        by_model.append(set(generator_rows.loc[generator_rows["model"] == model, "bg_id"]))
    common_ids = sorted(set.intersection(*by_model))
    if len(common_ids) < PILOT_BG_PER_GENERATOR:
        raise ValueError(
            f"{generator}: only {len(common_ids)} background IDs are common to all models; "
            f"the pilot requires {PILOT_BG_PER_GENERATOR}"
        )
    selected_bg_ids[generator] = sorted(
        int(value)
        for value in rng.choice(common_ids, size=PILOT_BG_PER_GENERATOR, replace=False)
    )

selected_mask = np.zeros(len(contrasts), dtype=bool)
for generator, background_ids in selected_bg_ids.items():
    selected_mask |= (
        contrasts["generator"].eq(generator)
        & contrasts["bg_id"].isin(background_ids)
    ).to_numpy()

pilot_all = contrasts.loc[selected_mask].copy()
live_pilot = (
    pilot_all[pilot_all["live"].astype(bool)]
    .sort_values(["model", "generator", "bg_id", "f_lock", "phase_idx"])
    .reset_index(drop=True)
)

coverage = (
    live_pilot.groupby(["model", "generator"], observed=True)
    .size()
    .rename("n_live")
    .reset_index()
)
if len(coverage) != len(expected_models) * len(expected_generators):
    raise ValueError("The selected live subset does not cover every model x generator cell")
if (coverage["n_live"] == 0).any():
    raise ValueError("At least one model x generator cell has no live observations")

print("source rows       :", len(contrasts))
print("selected rows     :", len(pilot_all))
print("selected live rows:", len(live_pilot))
print("selected bg IDs   :", selected_bg_ids)
display(coverage.pivot(index="model", columns="generator", values="n_live"))

del pilot_all, selected_mask
gc.collect()


### 5. Define the revised Model A

This is copied from the current operational notebook. The scientific response, priors and
configuration-level regression are unchanged. Only the sampling parametrisation is revised:

- configuration effects are sampled in the centred form because each level has many observations;
- harmonic and background offsets are constrained to sum to zero;
- the Student-t likelihood remains the same.

The zero-sum constraints prevent group means from exchanging an arbitrary constant with the
intercept. They do not force the individual effects to zero.


In [ ]:
PRIOR_SCALE = 0.5
NU = 4
MODEL_VERSION = "A-centred-zerosum-harm-bg-v1"


def codes(series: pd.Series) -> tuple[np.ndarray, np.ndarray]:
    integer_codes, levels = pd.factorize(series, sort=False)
    return np.asarray(integer_codes, dtype=int), np.asarray(levels)


def overlap_scaled(df: pd.DataFrame, configuration_levels: np.ndarray) -> np.ndarray:
    overlap = (
        df.groupby("model", observed=True)["overlap"]
        .first()
        .reindex(configuration_levels)
        .to_numpy(float)
    )
    return (overlap - overlap.mean()) / 0.5


def log_patch_centred(df: pd.DataFrame, configuration_levels: np.ndarray) -> np.ndarray:
    patch_size = (
        df.groupby("model", observed=True)["P"]
        .first()
        .reindex(configuration_levels)
        .to_numpy(float)
    )
    log_patch = np.log(patch_size)
    return log_patch - log_patch.mean()


def model_A_pilot(df: pd.DataFrame, scale: float = PRIOR_SCALE, nu: int = NU) -> pm.Model:
    configuration_code, configuration_levels = codes(df["model"])
    harmonic_code, harmonic_levels = codes(df["f_lock"].round(3).astype(str))
    background_code, background_levels = codes(
        df["generator"].astype(str) + "#" + df["bg_id"].astype(str)
    )
    overlap_t = overlap_scaled(df, configuration_levels)
    log_patch_t = log_patch_centred(df, configuration_levels)
    observed_contrast = df["d"].to_numpy(float)

    coordinates = {
        "config": configuration_levels,
        "harmonic": harmonic_levels,
        "background": background_levels,
        "obs": np.arange(len(observed_contrast)),
    }

    with pm.Model(coords=coordinates) as model:
        beta_bar = pm.StudentT("beta_bar", nu=nu, mu=0.0, sigma=scale)
        delta_O = pm.StudentT("delta_O", nu=nu, mu=0.0, sigma=scale)
        delta_P = pm.StudentT("delta_P", nu=nu, mu=0.0, sigma=scale)
        configuration_mean = beta_bar + delta_O * overlap_t + delta_P * log_patch_t

        tau = pm.HalfStudentT("tau", nu=nu, sigma=scale)
        beta = pm.Normal("beta", mu=configuration_mean, sigma=tau, dims="config")

        sigma_harm = pm.HalfStudentT("sigma_harm", nu=nu, sigma=scale)
        u_harm = pm.ZeroSumNormal("u_harm", sigma=sigma_harm, dims="harmonic")

        sigma_bg = pm.HalfStudentT("sigma_bg", nu=nu, sigma=scale)
        u_bg = pm.ZeroSumNormal("u_bg", sigma=sigma_bg, dims="background")

        location = beta[configuration_code] + u_harm[harmonic_code] + u_bg[background_code]
        sigma = pm.HalfStudentT("sigma", nu=nu, sigma=scale)
        pm.StudentT("d", nu=nu, mu=location, sigma=sigma, observed=observed_contrast, dims="obs")

        pm.Deterministic("recovery_ratio", pm.math.exp(beta_bar))

    return model


pilot_model = model_A_pilot(live_pilot)
print("model version:", MODEL_VERSION)
print("free variables:", [variable.name for variable in pilot_model.free_RVs])
print(
    "levels:",
    {
        "config": len(pilot_model.coords["config"]),
        "harmonic": len(pilot_model.coords["harmonic"]),
        "background": len(pilot_model.coords["background"]),
        "observations": len(pilot_model.coords["obs"]),
    },
)


### 6. Preflight and immutable pilot manifest

The manifest fingerprints the input table, exact selected background IDs, model version, sampling
settings, package versions, and repository revision. If any of these change, the notebook refuses
to reuse the old pilot checkpoint. Choose a new `PILOT_RUN_ID` for a different pilot.


In [ ]:
PILOT_ROOT.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = PILOT_ROOT / "pilot_manifest.json"
CHECKPOINT_PATH = PILOT_ROOT / "04_A_zerosum_pilot.nc"
DIAGNOSTICS_PATH = PILOT_ROOT / "05_A_pilot_diagnostics.parquet"
RESULT_PATH = PILOT_ROOT / "05_A_pilot_result.json"
TRACE_PATH = PILOT_ROOT / "A_pilot_trace.png"


def package_version(name: str):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return None


repository_revision = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
source_sha256 = cp.sha256_file(SOURCE_FILE)

pilot_spec = {
    "schema_version": 1,
    "non_reportable": True,
    "source_run_id": SOURCE_RUN_ID,
    "source_file": str(SOURCE_FILE),
    "source_sha256": source_sha256,
    "repository_revision": repository_revision,
    "model_version": MODEL_VERSION,
    "model_formula": (
        "d ~ StudentT(4, beta[config] + u_harm + u_bg, sigma); "
        "beta centred; u_harm and u_bg ZeroSumNormal"
    ),
    "prior_scale": PRIOR_SCALE,
    "nu": NU,
    "seed": SEED,
    "selected_bg_ids": selected_bg_ids,
    "selected_live_rows": int(len(live_pilot)),
    "sampling": {
        "draws": DRAWS,
        "tune": TUNE,
        "chains": CHAINS,
        "cores": CORES,
        "target_accept": TARGET_ACCEPT,
        "backend": NUTS_BACKEND,
        "log_likelihood": False,
    },
    "packages": {
        name: package_version(name)
        for name in ("pymc", "arviz", "nutpie", "numpy", "pandas", "pyarrow", "h5netcdf")
    },
    "python": platform.python_version(),
    "validation_only": VALIDATE_ONLY,
}
pilot_fingerprint = cp.fingerprint(pilot_spec)

if MANIFEST_PATH.is_file():
    pilot_manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    if pilot_manifest.get("pilot_fingerprint") != pilot_fingerprint:
        raise ValueError(
            "Existing pilot manifest does not match this run. Do not overwrite it; change "
            "PILOT_RUN_ID and run again."
        )
else:
    if CHECKPOINT_PATH.exists():
        raise ValueError("Untracked pilot checkpoint exists without a matching manifest")
    pilot_manifest = {
        "schema_version": 1,
        "pilot_fingerprint": pilot_fingerprint,
        "pilot_spec": pilot_spec,
        "status": "prepared",
        "artifacts": {},
    }
    cp.atomic_json(MANIFEST_PATH, pilot_manifest)

print("preflight: PASS")
print("input sha256:", source_sha256)
print("pilot fingerprint:", pilot_fingerprint)
print("checkpoint exists:", CHECKPOINT_PATH.is_file())
print("source namespace remains read-only:", SOURCE_ROOT)


### 7. Fit or resume the pilot

This is the only sampling cell. If a matching checkpoint already exists, it is loaded instead.
Sampling uses no observation-wise `log_likelihood`, so the checkpoint should remain small.
Interrupting this cell before the atomic save completes leaves no valid checkpoint and the pilot
must be sampled again.


In [ ]:
if CHECKPOINT_PATH.is_file():
    expected_hash = pilot_manifest.get("artifacts", {}).get(CHECKPOINT_PATH.name, {}).get("sha256")
    actual_hash = cp.sha256_file(CHECKPOINT_PATH)
    if not expected_hash or actual_hash != expected_hash:
        raise ValueError("Pilot checkpoint is untracked or its hash differs from the manifest")
    print("loading completed pilot checkpoint:", CHECKPOINT_PATH)
    idata_A_pilot = az.from_netcdf(CHECKPOINT_PATH)
    fit_seconds = pilot_manifest.get("fit_seconds")
else:
    sampling_kwargs = {}
    if NUTS_BACKEND != "pymc":
        sampling_kwargs["nuts_sampler"] = NUTS_BACKEND

    started = time.perf_counter()
    with pilot_model:
        idata_A_pilot = pm.sample(
            draws=DRAWS,
            tune=TUNE,
            chains=CHAINS,
            cores=CORES,
            random_seed=SEED,
            target_accept=TARGET_ACCEPT,
            progressbar=True,
            idata_kwargs={"log_likelihood": False},
            **sampling_kwargs,
        )
    fit_seconds = float(time.perf_counter() - started)

    cp.atomic_netcdf(CHECKPOINT_PATH, idata_A_pilot)
    checkpoint_entry = {
        "sha256": cp.sha256_file(CHECKPOINT_PATH),
        "bytes": CHECKPOINT_PATH.stat().st_size,
    }
    pilot_manifest.update(
        {
            "status": "sampled",
            "fit_seconds": fit_seconds,
            "artifacts": {**pilot_manifest.get("artifacts", {}), CHECKPOINT_PATH.name: checkpoint_entry},
        }
    )
    cp.atomic_json(MANIFEST_PATH, pilot_manifest)
    print("checkpoint saved atomically:", CHECKPOINT_PATH)

print("fit minutes:", None if fit_seconds is None else round(float(fit_seconds) / 60, 2))
print("checkpoint MiB:", round(CHECKPOINT_PATH.stat().st_size / 2**20, 2))
print("log_likelihood requested:", False)


## Checks

### 8. Diagnose chains and the remaining level decomposition

For this short pilot the ESS threshold is 400 rather than the final full-run threshold of 1,000.
R-hat must still be below 1.01 and divergences must remain zero. The derived combined location is
checked separately: if it converges while `beta_bar` does not, a level-exchange problem remains.

Definitions:

- **R-hat** measures agreement among chains; values near one are required.
- **ESS** is the effective number of independent samples after accounting for autocorrelation.
- **Divergences** are sampler warnings about difficult posterior geometry.
- **BFMI** checks whether the sampler explores posterior energy effectively; below 0.30 is a warning.


In [ ]:
diagnostic_columns = ["ess_bulk", "ess_tail", "r_hat"]
parameter_diagnostics = az.summary(
    idata_A_pilot,
    kind="diagnostics",
    round_to="none",
).reindex(columns=diagnostic_columns)
parameter_diagnostics[diagnostic_columns] = parameter_diagnostics[diagnostic_columns].apply(
    pd.to_numeric, errors="coerce"
)

divergences = int(np.asarray(idata_A_pilot.sample_stats["diverging"]).sum())
try:
    min_bfmi = float(np.nanmin(np.asarray(az.bfmi(idata_A_pilot), dtype=float)))
except Exception:
    min_bfmi = float("nan")

posterior = idata_A_pilot.posterior


def mean_over_levels(data_array):
    level_dimensions = [dimension for dimension in data_array.dims if dimension not in ("chain", "draw")]
    return data_array.mean(dim=level_dimensions) if level_dimensions else data_array


combined_location = (
    mean_over_levels(posterior["beta"])
    + mean_over_levels(posterior["u_harm"])
    + mean_over_levels(posterior["u_bg"])
)
combined_dataset = xr.Dataset({"combined_location": combined_location})
combined_diagnostic = az.summary(
    combined_dataset,
    kind="diagnostics",
    round_to="none",
).reindex(columns=diagnostic_columns)
combined_diagnostic[diagnostic_columns] = combined_diagnostic[diagnostic_columns].apply(
    pd.to_numeric, errors="coerce"
)

zero_sum_checks = {}
for variable in ("u_harm", "u_bg"):
    centred_draws = mean_over_levels(posterior[variable])
    zero_sum_checks[variable] = float(np.max(np.abs(np.asarray(centred_draws))))

finite_global = np.isfinite(parameter_diagnostics[diagnostic_columns].to_numpy(float)).all()
global_max_rhat = float(np.nanmax(parameter_diagnostics["r_hat"]))
global_min_ess_bulk = float(np.nanmin(parameter_diagnostics["ess_bulk"]))
global_min_ess_tail = float(np.nanmin(parameter_diagnostics["ess_tail"]))
pilot_global_ok = bool(
    finite_global
    and global_max_rhat < RHAT_MAX
    and global_min_ess_bulk > PILOT_ESS_MIN
    and global_min_ess_tail > PILOT_ESS_MIN
    and divergences == 0
)

headline_parameters = [
    "beta_bar",
    "delta_O",
    "delta_P",
    "tau",
    "sigma_harm",
    "sigma_bg",
    "sigma",
]
headline = parameter_diagnostics.reindex(headline_parameters)
headline_ok = bool(
    headline.notna().all().all()
    and (headline["r_hat"] < RHAT_MAX).all()
    and (headline["ess_bulk"] > PILOT_ESS_MIN).all()
    and (headline["ess_tail"] > PILOT_ESS_MIN).all()
    and divergences == 0
)

combined_row = combined_diagnostic.loc["combined_location"]
combined_ok = bool(
    float(combined_row["r_hat"]) < RHAT_MAX
    and float(combined_row["ess_bulk"]) > PILOT_ESS_MIN
    and float(combined_row["ess_tail"]) > PILOT_ESS_MIN
    and divergences == 0
)

print("\nHeadline diagnostics")
display(headline)
print("\nCombined fitted location")
display(combined_diagnostic)
print("\nWorst R-hat rows")
display(parameter_diagnostics.sort_values("r_hat", ascending=False).head(10))
print("\nLowest bulk ESS rows")
display(parameter_diagnostics.sort_values("ess_bulk").head(10))

print(
    {
        "pilot_global_ok": pilot_global_ok,
        "headline_ok": headline_ok,
        "combined_location_ok": combined_ok,
        "max_rhat": global_max_rhat,
        "min_ess_bulk": global_min_ess_bulk,
        "min_ess_tail": global_min_ess_tail,
        "divergences": divergences,
        "min_bfmi": min_bfmi,
        "max_abs_group_mean": zero_sum_checks,
    }
)

diagnostics_to_save = parameter_diagnostics.reset_index().rename(columns={"index": "parameter"})
cp.atomic_parquet(DIAGNOSTICS_PATH, diagnostics_to_save)

pilot_result = {
    "non_reportable": True,
    "pilot_fingerprint": pilot_fingerprint,
    "pilot_global_ok": pilot_global_ok,
    "headline_ok": headline_ok,
    "combined_location_ok": combined_ok,
    "max_rhat": global_max_rhat,
    "min_ess_bulk": global_min_ess_bulk,
    "min_ess_tail": global_min_ess_tail,
    "divergences": divergences,
    "min_bfmi": min_bfmi,
    "pilot_rhat_limit": RHAT_MAX,
    "pilot_ess_min": PILOT_ESS_MIN,
    "zero_sum_checks": zero_sum_checks,
    "fit_seconds": fit_seconds,
}
cp.atomic_json(RESULT_PATH, pilot_result)

pilot_manifest.update(
    {
        "status": "diagnosed",
        "pilot_result": pilot_result,
        "artifacts": {
            **pilot_manifest.get("artifacts", {}),
            DIAGNOSTICS_PATH.name: {
                "sha256": cp.sha256_file(DIAGNOSTICS_PATH),
                "bytes": DIAGNOSTICS_PATH.stat().st_size,
            },
            RESULT_PATH.name: {
                "sha256": cp.sha256_file(RESULT_PATH),
                "bytes": RESULT_PATH.stat().st_size,
            },
        },
    }
)
cp.atomic_json(MANIFEST_PATH, pilot_manifest)
print("diagnostics saved:", DIAGNOSTICS_PATH)
print("result saved     :", RESULT_PATH)


### 9. Trace plots

Each line is one MCMC chain. Good mixing appears as overlapping stationary bands without chains
occupying persistently different levels. These plots are diagnostics, not an H1 result.


In [ ]:
trace_variables = [
    "beta_bar",
    "delta_O",
    "delta_P",
    "tau",
    "sigma_harm",
    "sigma_bg",
    "sigma",
]

figure, axes = plt.subplots(len(trace_variables), 1, figsize=(12, 2.1 * len(trace_variables)), sharex=True)
for axis, variable in zip(axes, trace_variables):
    values = np.asarray(posterior[variable].transpose("chain", "draw"), dtype=float)
    for chain_index, chain_values in enumerate(values):
        axis.plot(chain_values, lw=0.7, alpha=0.75, label=f"chain {chain_index}")
    axis.set_ylabel(variable)
    axis.axhline(np.mean(values), color="black", lw=0.7, alpha=0.6)

axes[0].legend(ncol=min(CHAINS, 4), fontsize=8, loc="upper right")
axes[-1].set_xlabel("retained draw")
figure.suptitle("Model A zero-sum pilot traces, NON-REPORTABLE", y=1.002)
figure.tight_layout()
figure.savefig(TRACE_PATH, dpi=140, bbox_inches="tight")
plt.show()

pilot_manifest["artifacts"][TRACE_PATH.name] = {
    "sha256": cp.sha256_file(TRACE_PATH),
    "bytes": TRACE_PATH.stat().st_size,
}
cp.atomic_json(MANIFEST_PATH, pilot_manifest)
print("trace figure saved:", TRACE_PATH)


## Next steps

### 10. Route the result

This cell deliberately prints no supported/refuted/inconclusive verdict. A 20-background pilot
can decide whether the parametrisation is numerically promising, not whether H1 is scientifically
true. The full gate still requires all 200 backgrounds, 2,000 tune, 2,000 draws, ESS above 1,000,
PPC adequacy, and sensitivity checks.


In [ ]:
print("=" * 78)
print("PILOT ROUTE")
print("=" * 78)

if pilot_global_ok and headline_ok and combined_ok:
    print("PASS: the centred/zero-sum parametrisation is promising on the short pilot.")
    print("Next: prepare one full A-only refit on all 100 backgrounds per generator.")
    print("Do not interpret beta_bar scientifically and do not run A' or sensitivities yet.")
elif combined_ok and not headline_ok:
    print("FAIL: the fitted total location converges but headline components still do not.")
    print("This is consistent with a remaining level-decomposition problem.")
    print("Next: inspect chain traces and test a zero-sum configuration deviation before any full fit.")
else:
    print("FAIL: convergence problems extend beyond the old constant-exchange ridge.")
    print("Next: inspect the worst R-hat/ESS rows and traces; do not add draws or backgrounds yet.")

if np.isfinite(min_bfmi) and min_bfmi < 0.30:
    print("Additional warning: BFMI < 0.30 indicates poor posterior-energy exploration.")

print("\nNON-REPORTABLE PILOT: no H1 T/F/I verdict is produced.")
print("Send these files if detailed review is needed:")
print(" -", RESULT_PATH)
print(" -", DIAGNOSTICS_PATH)
print(" -", TRACE_PATH)
